# Dual-angle seismic interpolation with zero-inclusive slope ranges

This notebook trains the double-output PINNslope architecture on the SEAM double-angle data from `DeepWave-KAUST/PINNs_signal_separation-pub`. It uses a crop containing opposing and crossing events, keeps one trace out of every three, and constrains the two normalized slopes to `(0, 1)` and `(-1, 0)`. Zero is no longer separated by an artificial dead zone.

In [ ]:
from pathlib import Path
import json
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import torch

from pinnslope.dual_angle import DualAnglePINN
from pinnslope.dual_angle_training import (
    export_predictions, load_checkpoint, load_gather, prepare_interpolation_data,
    save_checkpoint, train_epoch, write_history, write_json,
)

repo_root = Path.cwd()
if not (repo_root / 'pinnslope').is_dir():
    repo_root = repo_root.parent
os.chdir(repo_root)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
print('Repository:', repo_root)
print('Device:', device)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Experiment configuration

The source file is `(t, x)=(900, 300)`. The selected crop is loaded as `(x, t)=(180, 450)`. `trace_step=3` means that only one third of the spatial traces contribute to the amplitude loss.

In [ ]:
epochs = int(os.environ.get('DUAL_ANGLE_EPOCHS', '2000'))
output_directory = Path(os.environ.get(
    'DUAL_ANGLE_OUTPUT', 'runs/seam_dual_angle_zero_nsub3'
))
config = {
    'data': 'data/dual_angle/seam_shot_15Hz.npy',
    'orientation': 't-x',
    'x_start': 80, 'x_stop': 260,
    't_start': 400, 't_stop': 850,
    'dx': 10.0, 'dt': 0.004,
    'trace_step': 3,
    'epochs': epochs,
    'grid_batch_size': 2048,
    'trace_batch_size': 2048,
    'data_weight': 1000.0,
    'learning_rate': 1e-3,
    'positive_slope': (0.0, 1.0),
    'negative_slope': (-1.0, 0.0),
    'checkpoint_every': 100,
    'seed': 42,
}
config

## Load, crop, and subsample the double-angle data

In [ ]:
gather = load_gather(
    config['data'], orientation=config['orientation'],
    x_slice=(config['x_start'], config['x_stop']),
    t_slice=(config['t_start'], config['t_stop']),
)
training_data = prepare_interpolation_data(gather, config['trace_step'], device)
observed = np.full_like(gather, np.nan)
observed[::config['trace_step']] = gather[::config['trace_step']]
print('Crop shape (x, t):', gather.shape)
print('Observed traces:', len(training_data['observed_trace_indices']))
print('Held-out traces:', int(training_data['missing_trace_mask'].sum().item()))

limit = np.percentile(np.abs(gather), 99)
fig, axes = plt.subplots(1, 2, figsize=(13, 6), constrained_layout=True)
axes[0].imshow(gather.T, cmap='gray', aspect='auto', vmin=-limit, vmax=limit)
axes[0].set_title('Dense SEAM double-angle crop')
axes[1].imshow(observed.T, cmap='gray', aspect='auto', vmin=-limit, vmax=limit)
axes[1].set_title('Observed traces: one out of three')
for axis in axes:
    axis.set_xlabel('x sample in crop')
    axis.set_ylabel('t sample in crop')
plt.show()

## Four-output architecture

The wavefield network predicts `phi_1` and `phi_2`; the slope network predicts `sigma_1` and `sigma_2`. With the selected shifts and scales, `sigma_1=sigmoid(y_1)` approaches zero from above and `sigma_2=-1+sigmoid(y_2)` approaches zero from below.

In [ ]:
model = DualAnglePINN(
    wavefield_hidden=(512, 512, 512, 512),
    slope_hidden=(8, 8, 8, 8),
    positional_encoding=(8, 32, 2),
    slope_shifts=(0.0, -1.0),
    slope_scales=(1.0, 1.0),
    device=device,
)
wavefield_optimizer = torch.optim.Adam(
    model.wavefield_network.parameters(), lr=config['learning_rate']
)
slope_optimizer = torch.optim.Adam(
    model.slope_network.parameters(), lr=config['learning_rate']
)
trace_batch_size = min(
    config['trace_batch_size'], training_data['observed_coordinates'].shape[0]
)
generator = torch.Generator(device='cuda' if device.startswith('cuda') else 'cpu')
generator.manual_seed(config['seed'])
print(model)
print('Trace batch size:', trace_batch_size)

## Train with checkpointing

Set `DUAL_ANGLE_RESUME` to a checkpoint path before executing the notebook to resume an interrupted run.

In [ ]:
output_directory.mkdir(parents=True, exist_ok=True)
checkpoint_directory = output_directory / 'checkpoints'
config_for_file = {**config, 'device': device, 'gather_shape': list(gather.shape),
                   'trace_batch_size': trace_batch_size}
write_json(output_directory / 'config.json', config_for_file)
history = []
start_epoch = 0
resume_path = os.environ.get('DUAL_ANGLE_RESUME')
if resume_path:
    start_epoch, history = load_checkpoint(
        resume_path, model, wavefield_optimizer, slope_optimizer, device
    )
    print('Resuming at epoch', start_epoch)

started = time.time()
for epoch in range(start_epoch, config['epochs']):
    losses = train_epoch(
        model, wavefield_optimizer, slope_optimizer,
        training_data['full_coordinates'],
        training_data['observed_coordinates'], training_data['observed_values'],
        config['grid_batch_size'], trace_batch_size, config['data_weight'], generator,
    )
    history.append({'epoch': epoch, **losses})
    if epoch % 10 == 0 or epoch == config['epochs'] - 1:
        print(
            f"epoch={epoch:04d} total={losses['total']:.5e} "
            f"data={losses['data']:.5e} physics={losses['physics']:.5e}"
        )
    if (epoch + 1) % config['checkpoint_every'] == 0 or epoch == config['epochs'] - 1:
        save_checkpoint(
            checkpoint_directory / f'epoch_{epoch + 1:04d}.pt', epoch, model,
            wavefield_optimizer, slope_optimizer, history, config_for_file,
        )
        save_checkpoint(
            checkpoint_directory / 'latest.pt', epoch, model,
            wavefield_optimizer, slope_optimizer, history, config_for_file,
        )
        write_history(output_directory / 'losses.csv', history)
print('Runtime (minutes):', (time.time() - started) / 60.0)

## Export and inspect reconstruction

In [ ]:
metrics = export_predictions(
    output_directory, model, training_data['full_coordinates'],
    training_data['target'], training_data['missing_trace_mask'], gather.shape, 8192,
)
reconstruction = np.load(output_directory / 'reconstruction.npy')
phi_1 = np.load(output_directory / 'phi_1.npy')
phi_2 = np.load(output_directory / 'phi_2.npy')
slope_1 = np.load(output_directory / 'slope_1.npy')
slope_2 = np.load(output_directory / 'slope_2.npy')
metrics

In [ ]:
figure_directory = output_directory / 'figures'
figure_directory.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(2, 2, figsize=(13, 11), constrained_layout=True)
fields = [gather, observed, reconstruction, reconstruction - gather]
titles = ['Dense reference', 'Observed 1/3 traces', 'Reconstruction', 'Error']
cmaps = ['gray', 'gray', 'gray', 'RdBu_r']
for axis, field, title, cmap in zip(axes.ravel(), fields, titles, cmaps):
    field_limit = np.percentile(np.abs(field[np.isfinite(field)]), 99)
    axis.imshow(field.T, cmap=cmap, aspect='auto', vmin=-field_limit, vmax=field_limit)
    axis.set_title(title)
    axis.set_xlabel('x sample in crop')
    axis.set_ylabel('t sample in crop')
fig.suptitle(f"Missing-trace MSE: {metrics['missing_mse']:.4e}")
fig.savefig(figure_directory / 'interpolation_overview.png', dpi=180)
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 11), constrained_layout=True)
component_limit = np.percentile(np.abs(np.concatenate([phi_1.ravel(), phi_2.ravel()])), 99)
for axis, field, title in zip(axes[0], [phi_1, phi_2], ['phi_1', 'phi_2']):
    image = axis.imshow(field.T, cmap='gray', aspect='auto',
                        vmin=-component_limit, vmax=component_limit)
    axis.set_title(title)
    fig.colorbar(image, ax=axis, shrink=0.8)
for axis, field, title in zip(axes[1], [slope_1, slope_2], ['sigma_1 > 0', 'sigma_2 < 0']):
    image = axis.imshow(field.T, cmap='viridis', aspect='auto')
    axis.set_title(title)
    fig.colorbar(image, ax=axis, shrink=0.8)
for axis in axes.ravel():
    axis.set_xlabel('x sample in crop')
    axis.set_ylabel('t sample in crop')
fig.savefig(figure_directory / 'components_and_slopes.png', dpi=180)
plt.show()

In [ ]:
loss_history = np.genfromtxt(output_directory / 'losses.csv', delimiter=',', names=True)
fig, axis = plt.subplots(figsize=(10, 5), constrained_layout=True)
axis.semilogy(loss_history['epoch'], loss_history['total'], label='total')
axis.semilogy(loss_history['epoch'], loss_history['data'], label='data')
axis.semilogy(loss_history['epoch'], loss_history['physics'], label='physics')
axis.set_xlabel('Epoch')
axis.set_ylabel('Loss')
axis.grid(True, alpha=0.25)
axis.legend()
fig.savefig(figure_directory / 'training_losses.png', dpi=180)
plt.show()